# AF2 complementary mechanisms — static audit
Audit AF2CTRL, AF2FS1, AF2SFS1, dan AF2BHCL1. Tidak melakukan training atau membuka test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, torch, traceback
from pathlib import Path

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(3):
    completed = subprocess.run(clone)
    if completed.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
else:
    raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('SETUP SELESAI:', BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

AF2_REL = 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'
PROJECT = resolve_drive_project_root(required_relative_paths=(AF2_REL,))
AF2 = require_project_artifact(PROJECT, AF2_REL)
OUTPUT = PROJECT / 'experiments/faruq-v3-af2-complement-v1'
STATIC = OUTPUT / 'static_audit.json'
print('PROJECT:', PROJECT)
print('AF2:', AF2)
print('STATIC:', STATIC)

In [ ]:
from coffee_detector.af2_complement.audit import run_af2_complement_static_audit

print('MENJALANKAN STATIC AUDIT', flush=True)
try:
    audit = run_af2_complement_static_audit(AF2, STATIC, device='0')
except Exception:
    traceback.print_exc()
    if STATIC.is_file():
        print('STATIC AUDIT YANG SEMPAT TERSIMPAN:')
        print(STATIC.read_text(errors='replace'))
    raise

parameters = {
    arm: {'total': row['parameters'], 'added': row['added_parameters']}
    for arm, row in audit['arms'].items()
}
print('PARAMETERS:', json.dumps(parameters, indent=2))
print('GATES:', json.dumps(audit['gates'], indent=2))
print('DECISION:', audit['decision'])
print('SAVED:', STATIC)
assert audit['decision'] == 'PASS', 'STOP: static audit gagal; jangan training.'
print('PASS: empat arm boleh dijalankan terpisah/paralel. Test tetap terkunci.')